In [1]:
import pandas as pd
user_category_df = pd.read_csv(r"D:\내 자료실(이동,삭제 절대금지~~!!)\Desktop\data analysis\Proj_1 e_cummerce\Python\user_category.csv")

In [2]:
user_category_df.head()

,user_id,category_id,product_cnt,event_cnt,purchase_flag,purchase_category_cnt
0,10300217,2053013563424899933,1,1,0,0
1,29515875,2053013554415534427,2,5,0,0
2,29515875,2053013557192163841,3,6,0,0
3,31198833,2053013553559896355,1,1,0,0
4,31198833,2053013554751078769,1,1,0,0


In [3]:
# user_id 별 category_cnt를 구하는 임시 데이터 프레임

tmp = (user_category_df
    .groupby("user_id")["category_id"]
    .nunique().rename("category_cnt")
    .reset_index())

In [4]:
# user_category_df 에 category_cnt 붙이기

summary=user_category_df.merge(tmp, on = "user_id", how = "left")

In [5]:
# category_group 기준잡기

tmp["category_cnt"].quantile([0.33, 0.67])

0.33    1.0
0.67    2.0
Name: category_cnt, dtype: float64

In [6]:
pd.set_option('display.max_rows', 200)

In [7]:
# category_count 극단찾기

tmp["category_cnt"].value_counts().sort_index(ascending=True)

category_cnt
1      1938281
2       615702
3       329445
4       207572
5       140623
6       100467
7        73658
8        55677
9        43208
10       33467
11       26397
12       21123
13       17336
14       13934
15       11423
16        9612
17        8047
18        6744
19        5726
20        4792
21        4074
22        3415
23        3001
24        2522
25        2190
26        1836
27        1686
28        1408
29        1281
30        1123
31         932
32         866
33         705
34         702
35         583
36         565
37         492
38         446
39         421
40         317
41         293
42         275
43         250
44         194
45         199
46         170
47         162
48         156
49         137
50         115
51         115
52          85
53          94
54         108
55          67
56          76
57          60
58          50
59          63
60          37
61          38
62          46
63          39
64          48
65          24
66          

In [8]:
# category_group 기준잡기2

summary["category_cnt"].quantile([0.33, 0.67])

0.33    3.0
0.67    8.0
Name: category_cnt, dtype: float64

In [9]:
# category_count 극단찾기2

summary["category_cnt"].value_counts().sort_index(ascending=True)

category_cnt
1      1938281
2      1231404
3       988335
4       830288
5       703115
6       602802
7       515606
8       445416
9       388872
10      334670
11      290367
12      253476
13      225368
14      195076
15      171345
16      153792
17      136799
18      121392
19      108794
20       95840
21       85554
22       75130
23       69023
24       60528
25       54750
26       47736
27       45522
28       39424
29       37149
30       33690
31       28892
32       27712
33       23265
34       23868
35       20405
36       20340
37       18204
38       16948
39       16419
40       12680
41       12013
42       11550
43       10750
44        8536
45        8955
46        7820
47        7614
48        7488
49        6713
50        5750
51        5865
52        4420
53        4982
54        5832
55        3685
56        4256
57        3420
58        2900
59        3717
60        2220
61        2318
62        2852
63        2457
64        3072
65        1560
66        17

In [10]:
# category_group 만들기

summary["category_group"] = pd.cut(
    summary["category_cnt"],
    bins=[0, 3, 8, 1000],
    labels=["1~3", "4~8", "9+"]
)

In [11]:
# rank_table 만들기 전 요약 프레임

rank_summary = (
    summary[summary["purchase_flag"] == 1]
    .groupby(["category_group", "category_id"], observed=True)
    .size()
    .reset_index(name="purchase_cnt")
)

In [12]:
# 구매비율 전환

rank_summary["purchase_pct"] = (rank_summary["purchase_cnt"] /
    rank_summary.groupby("category_group", observed=True)["purchase_cnt"].transform("sum"))

In [13]:
# 그룹별 rank 메기기

rank_summary["rank"] = (rank_summary
    .groupby("category_group", observed=True)["purchase_cnt"]
    .rank(method="first", ascending=False)
    .astype(int))

In [14]:
rank_summary.head()

,category_group,category_id,purchase_cnt,purchase_pct,rank
0,1~3,2053013552226107603,11,0.000041,312
1,1~3,2053013552259662037,613,0.002286,38
2,1~3,2053013552293216471,870,0.003244,29
3,1~3,2053013552326770905,1006,0.003752,26
4,1~3,2053013552351936731,175,0.000653,91


In [15]:
# category_group 별 상위 10위 추출

top10 = (rank_summary[rank_summary["rank"] <= 10]
    .sort_values(["category_group", "rank"]))

In [16]:
top10

,category_group,category_id,purchase_cnt,purchase_pct,rank
83,1~3,2053013555631882655,128235,0.478212,1
37,1~3,2053013553559896355,27099,0.101057,2
59,1~3,2053013554658804075,11805,0.044023,3
51,1~3,2053013554415534427,9829,0.036654,4
264,1~3,2053013563810775923,6235,0.023251,5
152,1~3,2053013558920217191,5776,0.021540,6
296,1~3,2053013565983425517,5487,0.020462,7
33,1~3,2053013553341792533,4359,0.016256,8
259,1~3,2053013563651392361,2941,0.010968,9
267,1~3,2053013563911439225,2586,0.009644,10


In [17]:
# pivot_table 만들기

top10_pivot = top10.pivot_table(
    index="rank",
    columns="category_group",
    values="category_id",
    aggfunc="first")

top10_pivot

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_4824\10995195.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  top10_pivot = top10.pivot_table(


category_group,1~3,4~8,9+
rank,,,
1,2053013555631882655,2053013555631882655,2053013555631882655
2,2053013553559896355,2053013553559896355,2053013553559896355
3,2053013554658804075,2053013554658804075,2053013554658804075
4,2053013554415534427,2053013554415534427,2053013554415534427
5,2053013563810775923,2053013565983425517,2053013565983425517
6,2053013558920217191,2053013563810775923,2053013563810775923
7,2053013565983425517,2053013553341792533,2053013563651392361
8,2053013553341792533,2053013558920217191,2053013553853497655
9,2053013563651392361,2053013563651392361,2053013558920217191
